# Классификация обратной связи пользователей интернет-магазина

- Автор: Березинский Вадим Сергеевич
- Дата: 01.08.2025

## Цели и задачи проекта

**Цель исследования:** автоматизировать обработку обратной связи пользователей интернет-магазина.

**Задачи:**
1. Познакомиться с данными, составить план решения задачи.
2. Разработать классификацию обратнй связи пользователей
3. Провести сегментацию отзывов согласно разработанной классификации.
4. Сформулировать выводы по проведённому анализу.

## Описание данных

Анализ проводится на основе данных интернет-магазина, представленных в таблице `dostavka`.

In [4]:
# Импортируем pandas для работы с данными:

import pandas as pd

# Импортируем библиотеки для визуализации:

import matplotlib.pyplot as plt
import seaborn as sns

## Знакомство с данными, предварительная обработка

In [6]:
# Познакомимся с данными
df=pd.read_csv('dostavka_202507311419.csv')
df.head(10)

,Unnamed: 0,Магазин,Номер заказа,Дата поступления,Статус,Оценка,Текст отзыва,Текст_класс_1,Текст_класс_2,Другое,is_p
0,0,store_00,42811R65031404,2011-06-19 00:00:56,Не проверен,5,NaN,NaN,NaN,NaN,NaN
1,1,store_01,143171885R0421,2011-06-19 00:05:42,Не проверен,5,NaN,NaN,NaN,NaN,NaN
2,2,store_02,50488R51918131,2011-06-19 00:08:19,Не проверен,5,NaN,NaN,NaN,NaN,NaN
3,3,store_03,4R556131941173,2011-06-19 00:13:57,Не проверен,3,долго заказ ждал... хотя доставку оплату повыс...,NaN,NaN,NaN,NaN
4,4,store_04,1117R271442037,2011-06-19 00:27:02,Не проверен,3,NaN,NaN,NaN,Без описания,NaN
5,5,store_05,3R111814004475,2011-06-19 00:30:14,Не проверен,1,"Персики дубовые , зуб можно сломать !!!!",NaN,NaN,NaN,NaN
6,6,store_06,91421174R35122,2011-06-19 00:31:25,Не проверен,5,NaN,NaN,NaN,NaN,NaN
7,7,store_06,711R4433421951,2011-06-19 00:31:52,Не проверен,5,NaN,NaN,NaN,NaN,NaN
8,8,store_07,781129116R1448,2011-06-19 00:48:18,Не проверен,5,NaN,NaN,NaN,NaN,NaN
9,9,store_08,035004185R1135,2011-06-19 00:50:05,Не проверен,1,"Здравствуйте! Ужасно! Я заказывала огурцы, огу...",NaN,NaN,NaN,NaN


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56874 entries, 0 to 56873
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        56874 non-null  int64  
 1   Магазин           56874 non-null  object 
 2   Номер заказа      56874 non-null  object 
 3   Дата поступления  56874 non-null  object 
 4   Статус            56874 non-null  object 
 5   Оценка            56874 non-null  int64  
 6   Текст отзыва      19271 non-null  object 
 7   Текст_класс_1     695 non-null    object 
 8   Текст_класс_2     0 non-null      float64
 9   Другое            6348 non-null   object 
 10  is_p              0 non-null      float64
dtypes: float64(2), int64(2), object(7)
memory usage: 4.8+ MB


Мы видим, что таблица `dostavka` состоит из 56874 строк и 11 столбцов. При этом столбец `Unnamed: 0` дублирует индексы и не требуется для дальнейшей работы, столбцы `Текст_класс_2` и `is_p` полностью пустые и неинформативные, столбцы `Текст_класс_1` и `Другое` содержат много пустых значение, а заполненные значения повторяют значения столбца `Текст отзыва`. Предлагаем удалить неинформативные столбцы из датафрейма.

In [9]:
# Удалим неинформативные столбцы:

df = df.drop(['Unnamed: 0', 'Текст_класс_1', 'Текст_класс_2', 'Другое', 'is_p'], axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56874 entries, 0 to 56873
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Магазин           56874 non-null  object
 1   Номер заказа      56874 non-null  object
 2   Дата поступления  56874 non-null  object
 3   Статус            56874 non-null  object
 4   Оценка            56874 non-null  int64 
 5   Текст отзыва      19271 non-null  object
dtypes: int64(1), object(5)
memory usage: 2.6+ MB


В итоге мы получили таблицу из 6 столбцов и 56874 строк. Все столбцы заполнены полностью, кроме столбца `Текст отзыва`. Пропуски в этом столбце обусловлены тем, что далеко не все покупатели оставляют текстовый отзыв на заказ.

## Разработка классификации обратнй связи пользователей

Поскольку нам необходимо провести анализ обратной связи пользователей и разработать ее классификацию, то в изучим содержание столбцов `Оценка` и `Текст отзыва`. В столбце `Оценка` мы ожидаем увидеть числовую оценку заказа по шкале от 1 до 5 без пропусков. Столбец `Текст отзыва` позволит нам разделить все претензии пользователей на группы.

In [12]:
# Выведем уникальные значения столбца `Оценка`:

df['Оценка'].unique()

array([5, 3, 1, 2, 4], dtype=int64)

Мы видим, что столбец `Оценка` действительно содержит числовую оценку заказа по шкале от 1 до 5 без пропусков.

In [14]:
# Выведем пятьдесят непустых значений столбца `Текст отзыва` и посмотрим, что пишут покупатели в отзывах.

# Убираем ограничение на ширину отображения текста в столбцах
pd.set_option('display.max_colwidth', None)

feedback = df['Текст отзыва'].dropna()

feedback.head(50)

3                                                                                                                                                                                                                                                                                                 долго заказ ждал... хотя доставку оплату повысили качество так себе
5                                                                                                                                                                                                                                                                                                                            Персики дубовые , зуб можно сломать !!!!
9                                                                                                                                                                                                                      Здравствуйте! Ужасно! Я заказывала огурцы, огурцов в доставке не оказ

Смысловой анализ текстов отзывов позволяет определить, что недовольство покупателей связано в первую очередь с качеством приобретенных продуктов, длительным временем ожидания заказа, неаккуратной доставкой товара курьером, неправильной комплектацией (сборкой) заказа, некорректным начислением или списанием бонусов, некорректными расчетами за купленный товар.

Исходя из этого, можно предложить следующую классификацию проблем сервиса, на которые обращают внимание покупатели в текстовых отзывах:
1. Некачественный товар
2. Неправильная сборка заказа
3. Длительное ожидание доставки
4. Неаккуратная доставка товара
5. Ошибка с начислением и списанием баллов
6. Некорректный расчёт с покупателем

## Сегментация отзывов согласно разработанной классификации

Можно предложить следующий вариант автоматизации обработки текстовых отзывов покупателей:

1. Заказы с оценкой 5 (значение 5 в столбце 'Оценка') независимо от наличия любого текста в столбце 'Текст отзыва' считать положительно оценнными пользователями и размечать как 'Все хорошо'.

2. Заказы с оценкой ниже 5 и пустым значением в столбце 'Текст отзыва' размечать как 'Нет отзыва'.

3. Заказы с оценкой ниже 5 и непустыми значениями в столбце 'Текст отзыва' размечать согласно предложенной классификации проблем. В этих целях для каждой проблемы подобрать набор ключевых слов, при наличии которых в тексте отзыва строка будет размечаться соответствующим значением из классификации.

4. Оставшиеся неразмеченными отзывы размечать как 'Требует обработки'. Эти отзывы потребуют ручной обработки. В результате ручной обработки могут выявляться дополнительные ключевые слова, позволяющие увеличить процент автоматического разбора отзывов.

In [17]:
# Задаём словари ключевых слов для каждой проблемы
keyword_map = {
    'Некачественный товар': [
        'тухл', 'слом', 'просроч', 'годност',
        'гнил'
    ],
    'Неправильная сборка заказа': [
        'не тот', 'перепутал', 'не довезли', 'не привезли',
        'вес', 'положили', 'вместо', 'путают', 'не оказалось'
    ],
    'Длительное ожидание доставки': [
        'ждал', 'долго', 'задерж', 'опозд', 'не успели',
        'часов'
    ],
    'Неаккуратная доставка товара': [
        'поврежден', 'помят', 'грязн', 'запачкан'
    ],
    'Ошибка с начислением и списанием баллов': [
        'балл', 'начислен', 'списан', 'бонус', 'кэшбэк',
        'скидк'
    ],
    'Некорректный расчёт с покупателем': [
        'деньги', 'вернут', 'сумм'
    ]
}


In [18]:
# Определим функцию 'classify_feedback', которая на вход получает одну строку (row).

def classify_feedback (row):

# Из каждой строки извлекается значение оценки и записывается в переменную rating:

    rating = row['Оценка']

# Из каждой строки извлекается текст из столбца 'Текст отзыва' и записывается в переменную text:

    text = row['Текст отзыва']

# Первым шагом запускаем цикл, который отберет строки с оценкой '5' и присвоит им значение 'Все хорошо':
    if rating == 5:
        return 'Все хорошо'

# Вторым шагом запускаем цикл, который отберет строки с оценкой меньше '5' и незаполненным текстом отзыва и присвоит им значение 'Нет отзыва':
    if rating < 5 and pd.isna(text):
        return 'Нет отзыва'

# Третьим шагом запускаем цикл, который отберет строки с оценкой меньше '5' и заполненным текстовым отзывом
    if rating < 5 and pd.notna(text):

# Текст отзыва приведем к нижнему регистру, чтобы поиск ключевых слов был нечувствителен к регистру.

        lower_text = text.lower()

# Создаём пустой список для хранения всех найденных категорий

        categories = []

# Задаем цикл, который проходит по словарям ключевых слов (label - это категория проблемы, keywords - список ключевых слов)

        for label, keywords in keyword_map.items():

# и для каждого ключевого слова из списка ключевых слов

            for kw in keywords:

# проверяет, есть ли это слово в тексте отзыва, приведенном к нижнему регистру (lower_text)

                if kw in lower_text:

# ключевое слово встречается в тексте отзыва, то категория проблемы (label) добавляется в список categories

                    categories.append(label)

# Как только найдено первое совпадение для категории, исполнение вложенного цикла прекращается

                    break
# Если список категорий не пустой, еще одним циклом соединяем все катгории в одну строку через запятую
        if categories:
            return ', '.join(categories)

# На четвертом шаге всем остальным строкам присваиваем значение 'Требует обработки'
    return 'Требует обработки'


In [19]:
# Создаем новый столбец 'Категория' и заполняем его с помощью функции classify_feedback

df['Категория'] = df.apply(classify_feedback, axis=1)

In [20]:
df.head(30)

,Магазин,Номер заказа,Дата поступления,Статус,Оценка,Текст отзыва,Категория
0,store_00,42811R65031404,2011-06-19 00:00:56,Не проверен,5,NaN,Все хорошо
1,store_01,143171885R0421,2011-06-19 00:05:42,Не проверен,5,NaN,Все хорошо
2,store_02,50488R51918131,2011-06-19 00:08:19,Не проверен,5,NaN,Все хорошо
3,store_03,4R556131941173,2011-06-19 00:13:57,Не проверен,3,долго заказ ждал... хотя доставку оплату повысили качество так себе,Длительное ожидание доставки
4,store_04,1117R271442037,2011-06-19 00:27:02,Не проверен,3,NaN,Нет отзыва
5,store_05,3R111814004475,2011-06-19 00:30:14,Не проверен,1,"Персики дубовые , зуб можно сломать !!!!",Некачественный товар
6,store_06,91421174R35122,2011-06-19 00:31:25,Не проверен,5,NaN,Все хорошо
7,store_06,711R4433421951,2011-06-19 00:31:52,Не проверен,5,NaN,Все хорошо
8,store_07,781129116R1448,2011-06-19 00:48:18,Не проверен,5,NaN,Все хорошо
9,store_08,035004185R1135,2011-06-19 00:50:05,Не проверен,1,"Здравствуйте! Ужасно! Я заказывала огурцы, огурцов в доставке не оказалось!! Чеков нет !! Доставку задержали, ужасно недовольна вашим сервисом","Неправильная сборка заказа, Длительное ожидание доставки"


In [21]:
# Проверим, какой процент отзывов требует ручной обработки после запуска автоматизации:

percentage_of_unprocessed = df[df['Категория']=='Требует обработки'].shape[0] / df.shape[0] * 100

print(f'Процент отзывов, требующих ручной обработки после запуска автоматизации, составляет {percentage_of_unprocessed:.2f}%.')

Процент отзывов, требующих ручной обработки после запуска автоматизации, составляет 11.13%.


In [23]:
# Создадим еще один датафрейм, в котором развернем списки категорий на несколько строк (по одной категории на строку) методом explode для возможности группировки и визуализации:

df['category'] = df['Категория'].str.split(', ')

df_exploded = df.explode('category')

## Выводы по проведённому анализу

На основе проведенного анализа данных об отзывах покупателей интернет-магазина предложена следующая классификация проблем сервиса, на которые обращают внимание покупатели в текстовых отзывах:
1. Некачественный товар
2. Неправильная сборка заказа
3. Длительное ожидание доставки
4. Неаккуратная доставка товара
5. Ошибка с начислением и списанием баллов
6. Некорректный расчёт с покупателем

Предложен следующий алгоритм автоматизации обработки текстовых отзывов покупателей:

1. Заказы с оценкой 5 независимо от наличия любого текста в столбце 'Текст отзыва' считать положительно оценнными пользователями и размечать как 'Все хорошо'.

2. Заказы с оценкой ниже 5 и пустым значением в столбце 'Текст отзыва' размечать как 'Нет отзыва'.

3. Заказы с оценкой ниже 5 и непустыми значениями в столбце 'Текст отзыва' размечать согласно предложенной классификации проблем. В этих целях для каждой проблемы подобрать набор ключевых слов, при наличии которых в тексте отзыва строка будет размечаться соответствующим значением из классификации.

4. Оставшиеся неразмеченными отзывы размечать как 'Требует обработки'. Эти отзывы потребуют ручной обработки. В результате ручной обработки могут выявляться дополнительные ключевые слова, позволяющие увеличить процент автоматического разбора отзывов.

На основании анализа пятидесяти непустых текстовых отзывов составлен словарь ключевых слов, позволяющий классифицировать текстовые отзывы по категориям проблем.

Разработана функция, сегментирующая отзывы покупателей в соответствии с предложенной классификацией.

После запуска предложенного варианта автоматизации процент отзывов, требующих ручной обработки, составил 11.13%. Далее в процессе обработки этих отзывов возможно дополнение словарей ключевых слов, что позволит дополнительно увеличить процент автоматически обрабатываемых отзывов.